In [4]:
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

In [5]:
medicare = pd.read_csv('Medicare_Hospital_Spending_by_Claim.csv',
                      dtype = {'Facility ID': str},
                      low_memory = False )

In [6]:
medicare.head()

,Facility Name,Facility ID,State,Period,Claim Type,Avg Spndg Per EP Hospital,Avg Spndg Per EP State,Avg Spndg Per EP National,Percent of Spndg Hospital,Percent of Spndg State,Percent of Spndg National,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,1 to 3 days Prior to Index Hospital Admission,Home Health Agency,25,22,19,0.09%,0.08%,0.07%,01/01/2024,12/31/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,1 to 3 days Prior to Index Hospital Admission,Hospice,2,2,1,0.01%,0.01%,0.00%,01/01/2024,12/31/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,1 to 3 days Prior to Index Hospital Admission,Inpatient,28,45,46,0.10%,0.17%,0.17%,01/01/2024,12/31/2024
3,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,1 to 3 days Prior to Index Hospital Admission,Outpatient,116,135,208,0.43%,0.52%,0.76%,01/01/2024,12/31/2024
4,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,1 to 3 days Prior to Index Hospital Admission,Skilled Nursing Facility,9,14,17,0.03%,0.05%,0.06%,01/01/2024,12/31/2024


In [7]:
print(medicare.isnull().sum())
print(medicare.dtypes)

Facility Name                0
Facility ID                  0
State                        0
Period                       0
Claim Type                   0
Avg Spndg Per EP Hospital    0
Avg Spndg Per EP State       0
Avg Spndg Per EP National    0
Percent of Spndg Hospital    0
Percent of Spndg State       0
Percent of Spndg National    0
Start Date                   0
End Date                     0
dtype: int64
Facility Name                object
Facility ID                  object
State                        object
Period                       object
Claim Type                   object
Avg Spndg Per EP Hospital     int64
Avg Spndg Per EP State        int64
Avg Spndg Per EP National     int64
Percent of Spndg Hospital    object
Percent of Spndg State       object
Percent of Spndg National    object
Start Date                   object
End Date                     object
dtype: object


In [9]:
medicare.columns = (
    medicare.columns 
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace(r'[^\w]', '_', regex = True)
)

print(medicare.columns.tolist())

['facility_name', 'facility_id', 'state', 'period', 'claim_type', 'avg_spndg_per_ep_hospital', 'avg_spndg_per_ep_state', 'avg_spndg_per_ep_national', 'percent_of_spndg_hospital', 'percent_of_spndg_state', 'percent_of_spndg_national', 'start_date', 'end_date']


In [10]:
medicare['facility_id'] = (
    medicare['facility_id']
    .str.strip()
    .str.zfill(6)
)


In [12]:
updated_cols =['percent_of_spndg_hospital','percent_of_spndg_state','percent_of_spndg_national']
medicare[updated_cols] = medicare[updated_cols].replace('%', '', regex=True).astype(float)

print(medicare[updated_cols])

       percent_of_spndg_hospital  percent_of_spndg_state  \
0                           0.09                    0.08   
1                           0.01                    0.01   
2                           0.10                    0.17   
3                           0.43                    0.52   
4                           0.03                    0.05   
...                          ...                     ...   
63641                       2.24                    3.08   
63642                      11.62                   11.28   
63643                       0.18                    0.60   
63644                       7.89                    6.81   
63645                     100.00                  100.00   

       percent_of_spndg_national  
0                           0.07  
1                           0.00  
2                           0.17  
3                           0.76  
4                           0.06  
...                          ...  
63641                       3.86  

In [13]:
print(medicare.dtypes)

facility_name                 object
facility_id                   object
state                         object
period                        object
claim_type                    object
avg_spndg_per_ep_hospital      int64
avg_spndg_per_ep_state         int64
avg_spndg_per_ep_national      int64
percent_of_spndg_hospital    float64
percent_of_spndg_state       float64
percent_of_spndg_national    float64
start_date                    object
end_date                      object
dtype: object


In [14]:
import numpy as np

print(f"Shape: {medicare.shape}")
print(f"Unique hospitals: {medicare['facility_id'].nunique()}")

CMS_NULLS = {
    'Not Available': np.nan,
    'Not Applicable': np.nan,
    'Too Few to Report': np.nan,
    'N/A': np.nan
}
medicare = medicare.replace(CMS_NULLS)

print(f"Nulls after replace:\n{medicare.isnull().sum()}")

Shape: (63646, 13)
Unique hospitals: 2893
Nulls after replace:
facility_name                0
facility_id                  0
state                        0
period                       0
claim_type                   0
avg_spndg_per_ep_hospital    0
avg_spndg_per_ep_state       0
avg_spndg_per_ep_national    0
percent_of_spndg_hospital    0
percent_of_spndg_state       0
percent_of_spndg_national    0
start_date                   0
end_date                     0
dtype: int64


In [15]:
dupes = medicare.duplicated(
    subset=['facility_id', 'period', 'claim_type']
).sum()
print(f"True duplicates: {dupes}")

True duplicates: 0


In [16]:
cols_to_drop = [
    'facility_name',  
    'state',          
    'start_date',     
    'end_date'        
]

medicare = medicare.drop(columns=cols_to_drop, errors='ignore')

print(f"Shape after drop: {medicare.shape}")
print(medicare.columns.tolist())

Shape after drop: (63646, 9)
['facility_id', 'period', 'claim_type', 'avg_spndg_per_ep_hospital', 'avg_spndg_per_ep_state', 'avg_spndg_per_ep_national', 'percent_of_spndg_hospital', 'percent_of_spndg_state', 'percent_of_spndg_national']


In [17]:
print(f"Unique hospitals : {medicare['facility_id'].nunique()}")
print(f"\nPeriods available:")
print(medicare['period'].value_counts())
print(f"\nClaim types available:")
print(medicare['claim_type'].value_counts())

Unique hospitals : 2893

Periods available:
period
1 to 3 days Prior to Index Hospital Admission                      20251
During Index Hospital Admission                                    20251
1 through 30 days After Discharge from Index Hospital Admission    20251
Complete Episode                                                    2893
Name: count, dtype: int64

Claim types available:
claim_type
Home Health Agency           8679
Hospice                      8679
Inpatient                    8679
Outpatient                   8679
Skilled Nursing Facility     8679
Durable Medical Equipment    8679
Carrier                      8679
Total                        2893
Name: count, dtype: int64


In [18]:

medicare['pivot_key'] = (
    medicare['period'].str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w]', '_', regex=True)
    + '__' +
    medicare['claim_type'].str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w]', '_', regex=True)
)

medicare_wide = medicare.pivot_table(
    index='facility_id',
    columns='pivot_key',
    values='avg_spndg_per_ep_hospital',
    aggfunc='first'
).reset_index()

medicare_wide.columns = (
    ['facility_id'] +
    ['med_' + col for col in medicare_wide.columns[1:]]
)

print(f"Wide table shape: {medicare_wide.shape}")
medicare_wide.head()

Wide table shape: (2893, 23)


,facility_id,med_1_through_30_days_after_discharge_from_index_hospital_admission__carrier,med_1_through_30_days_after_discharge_from_index_hospital_admission__durable_medical_equipment,med_1_through_30_days_after_discharge_from_index_hospital_admission__home_health_agency,med_1_through_30_days_after_discharge_from_index_hospital_admission__hospice,med_1_through_30_days_after_discharge_from_index_hospital_admission__inpatient,med_1_through_30_days_after_discharge_from_index_hospital_admission__outpatient,med_1_through_30_days_after_discharge_from_index_hospital_admission__skilled_nursing_facility,med_1_to_3_days_prior_to_index_hospital_admission__carrier,med_1_to_3_days_prior_to_index_hospital_admission__durable_medical_equipment,...,med_1_to_3_days_prior_to_index_hospital_admission__outpatient,med_1_to_3_days_prior_to_index_hospital_admission__skilled_nursing_facility,med_complete_episode__total,med_during_index_hospital_admission__carrier,med_during_index_hospital_admission__durable_medical_equipment,med_during_index_hospital_admission__home_health_agency,med_during_index_hospital_admission__hospice,med_during_index_hospital_admission__inpatient,med_during_index_hospital_admission__outpatient,med_during_index_hospital_admission__skilled_nursing_facility
0,010001,1548,160,979,267,5754,809,2834,878,6,...,116,9,26979,1446,30,0,0,12086,0,0
1,010005,1394,170,800,298,2679,1051,5697,630,8,...,26,14,23705,1079,7,0,0,9821,0,0
2,010006,1522,117,924,274,4677,792,4009,770,9,...,181,17,25745,1277,19,0,0,11093,0,0
3,010007,1270,82,959,293,3614,577,5112,553,14,...,43,13,20397,465,9,0,0,7345,0,0
4,010008,1275,63,1063,643,2180,1203,6772,480,6,...,201,0,21453,387,40,0,0,7043,0,0


In [19]:
medicare_wide.to_csv('medicare_cleaned.csv', index=False)
print("Saved as medicare_cleaned.csv")

Saved as medicare_cleaned.csv


In [20]:
print(medicare_wide.columns.tolist())

['facility_id', 'med_1_through_30_days_after_discharge_from_index_hospital_admission__carrier', 'med_1_through_30_days_after_discharge_from_index_hospital_admission__durable_medical_equipment', 'med_1_through_30_days_after_discharge_from_index_hospital_admission__home_health_agency', 'med_1_through_30_days_after_discharge_from_index_hospital_admission__hospice', 'med_1_through_30_days_after_discharge_from_index_hospital_admission__inpatient', 'med_1_through_30_days_after_discharge_from_index_hospital_admission__outpatient', 'med_1_through_30_days_after_discharge_from_index_hospital_admission__skilled_nursing_facility', 'med_1_to_3_days_prior_to_index_hospital_admission__carrier', 'med_1_to_3_days_prior_to_index_hospital_admission__durable_medical_equipment', 'med_1_to_3_days_prior_to_index_hospital_admission__home_health_agency', 'med_1_to_3_days_prior_to_index_hospital_admission__hospice', 'med_1_to_3_days_prior_to_index_hospital_admission__inpatient', 'med_1_to_3_days_prior_to_index_